# Обзор результатов HMM grid search

Ноутбук читает результаты `train_hmm_grid_search.py` из S3, проверяет полноту прогона, показывает ошибки, сравнивает конфигурации по `BIC`, `AIC` и `log_likelihood`, а также разбирает режимы лучших моделей.

In [ ]:
from __future__ import annotations

import io
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from botocore.exceptions import ClientError

PROJECT_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "build_price_feature_day.py").exists()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from build_price_feature_day import make_s3_client

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
plt.style.use("seaborn-v0_8-whitegrid")

## Конфигурация

Если `RUN_ID = None`, ноутбук найдёт последний `run_id` по `all_results.parquet`. Если итоговый файл не был собран, можно указать `RUN_ID` вручную: тогда ноутбук попробует собрать таблицу из отдельных `models/.../result.parquet`.

In [ ]:
S3_BUCKET = "binance-data-downloader"
OUTPUT_PREFIX = "features/hmm_grid_search/results"

# Пример: RUN_ID = "hmm_20260703". Оставьте None, чтобы взять самый свежий run.
RUN_ID = None

EXPECTED_MODELS = 600
TOP_N = 15

s3 = make_s3_client()

## S3 helpers

In [ ]:
def s3_key_exists(bucket: str, key: str) -> bool:
    try:
        s3.head_object(Bucket=bucket, Key=key)
        return True
    except ClientError as exc:
        code = exc.response.get("Error", {}).get("Code")
        if code in {"404", "NoSuchKey", "NotFound"}:
            return False
        raise


def list_s3_keys(bucket: str, prefix: str) -> list[str]:
    keys = []
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix.rstrip("/") + "/"):
        keys.extend(item["Key"] for item in page.get("Contents", []))
    return keys


def read_s3_parquet(bucket: str, key: str) -> pd.DataFrame:
    body = s3.get_object(Bucket=bucket, Key=key)["Body"].read()
    return pd.read_parquet(io.BytesIO(body))


def all_results_key(run_id: str) -> str:
    return f"{OUTPUT_PREFIX.strip('/')}/runs/run_id={run_id}/all_results.parquet"


def models_prefix(run_id: str) -> str:
    return f"{OUTPUT_PREFIX.strip('/')}/runs/run_id={run_id}/models"


def discover_run_ids() -> pd.DataFrame:
    keys = list_s3_keys(S3_BUCKET, f"{OUTPUT_PREFIX.strip('/')}/runs")
    rows = []
    for key in keys:
        marker = "/run_id="
        if marker not in key:
            continue
        run_id = key.split(marker, 1)[1].split("/", 1)[0]
        rows.append(
            {
                "run_id": run_id,
                "key": key,
                "is_all_results": key.endswith("/all_results.parquet"),
                "is_model_result": key.endswith("/result.parquet"),
            }
        )
    if not rows:
        return pd.DataFrame(columns=["run_id", "has_all_results", "model_files"])
    frame = pd.DataFrame(rows)
    return (
        frame.groupby("run_id", as_index=False)
        .agg(has_all_results=("is_all_results", "any"), model_files=("is_model_result", "sum"))
        .sort_values("run_id", ascending=False)
        .reset_index(drop=True)
    )

## Загрузка результатов

In [ ]:
runs = discover_run_ids()
runs.head(20)

In [ ]:
def load_results(run_id: str | None) -> tuple[str, pd.DataFrame]:
    if run_id is None:
        candidates = runs[runs["has_all_results"]]
        if candidates.empty:
            candidates = runs[runs["model_files"].gt(0)]
        if candidates.empty:
            raise FileNotFoundError(f"No HMM result runs found under s3://{S3_BUCKET}/{OUTPUT_PREFIX}/runs/")
        run_id = candidates.iloc[0]["run_id"]

    final_key = all_results_key(run_id)
    if s3_key_exists(S3_BUCKET, final_key):
        return run_id, read_s3_parquet(S3_BUCKET, final_key)

    result_keys = [key for key in list_s3_keys(S3_BUCKET, models_prefix(run_id)) if key.endswith("/result.parquet")]
    if not result_keys:
        raise FileNotFoundError(f"No model result files found for run_id={run_id}")
    frames = [read_s3_parquet(S3_BUCKET, key) for key in sorted(result_keys)]
    return run_id, pd.concat(frames, ignore_index=True)


run_id, results_raw = load_results(RUN_ID)
print(f"Loaded run_id={run_id}")
print(f"Rows: {len(results_raw):,}")
results_raw.head()

In [ ]:
results = results_raw.copy()

numeric_columns = [
    "n_components",
    "random_state",
    "n_rows",
    "train_seconds",
    "n_iter",
    "log_likelihood",
    "aic",
    "bic",
]
for column in numeric_columns:
    if column in results.columns:
        results[column] = pd.to_numeric(results[column], errors="coerce")

results["success"] = results["success"].astype(bool)
if "converged" in results.columns:
    results["converged"] = results["converged"].astype("boolean")

sort_columns = ["model_group", "n_components", "covariance_type", "random_state"]
results = results.sort_values(sort_columns).reset_index(drop=True)
successful = results[results["success"]].copy()
failed = results[~results["success"]].copy()

print(f"Expected models: {EXPECTED_MODELS}")
print(f"Actual rows: {len(results)}")
print(f"Success: {len(successful)}")
print(f"Failed: {len(failed)}")
print(f"Missing rows vs expected: {EXPECTED_MODELS - len(results)}")

## Общая сводка

In [ ]:
coverage = (
    results.groupby(["model_group", "n_components", "covariance_type"], dropna=False)
    .agg(models=("random_state", "count"), successes=("success", "sum"), failures=("success", lambda x: int((~x).sum())))
    .reset_index()
)
coverage

In [ ]:
runtime_summary = (
    successful.groupby(["model_group", "covariance_type"], dropna=False)
    .agg(
        models=("random_state", "count"),
        total_hours=("train_seconds", lambda x: x.sum() / 3600),
        median_seconds=("train_seconds", "median"),
        p90_seconds=("train_seconds", lambda x: x.quantile(0.90)),
        max_seconds=("train_seconds", "max"),
        converged_rate=("converged", "mean"),
        median_iter=("n_iter", "median"),
    )
    .reset_index()
)
runtime_summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4), constrained_layout=True)

status_counts = results["success"].map({True: "success", False: "failed"}).value_counts()
axes[0].bar(status_counts.index, status_counts.values, color=["#54A24B", "#E45756"][: len(status_counts)])
axes[0].set_title("Run status")
axes[0].set_ylabel("Models")

group_counts = results.groupby("model_group")["success"].sum().reindex(["price", "flow", "market"])
axes[1].bar(group_counts.index, group_counts.values, color="#4C78A8")
axes[1].set_title("Successful models by group")
axes[1].set_ylabel("Successes")

if len(successful):
    axes[2].hist(successful["train_seconds"].dropna(), bins=40, color="#F58518", alpha=0.75)
axes[2].set_title("Training time distribution")
axes[2].set_xlabel("Seconds")
axes[2].set_ylabel("Models")

plt.show()

## Ошибки и несходимость

In [ ]:
failed[["model_group", "n_components", "covariance_type", "random_state", "error"]].head(50)

In [ ]:
if len(failed):
    display(
        failed.assign(error_short=failed["error"].astype(str).str.slice(0, 220))
        .groupby("error_short", dropna=False)
        .size()
        .rename("count")
        .reset_index()
        .sort_values("count", ascending=False)
    )
else:
    print("No failed configurations")

not_converged = successful[successful["converged"].eq(False)]
print(f"Not converged successful fits: {len(not_converged)}")
not_converged[["model_group", "n_components", "covariance_type", "random_state", "n_iter", "bic", "aic"]].head(50)

## Лучшие конфигурации

In [ ]:
metric_columns = [
    "model_group",
    "n_components",
    "covariance_type",
    "random_state",
    "log_likelihood",
    "aic",
    "bic",
    "converged",
    "n_iter",
    "train_seconds",
    "n_rows",
]

best_by_bic = successful.loc[successful.groupby("model_group")["bic"].idxmin(), metric_columns].sort_values("model_group")
best_by_aic = successful.loc[successful.groupby("model_group")["aic"].idxmin(), metric_columns].sort_values("model_group")
best_by_ll = successful.loc[successful.groupby("model_group")["log_likelihood"].idxmax(), metric_columns].sort_values("model_group")

best_by_bic

In [ ]:
print("Best by AIC")
display(best_by_aic)

print("Best by log-likelihood")
display(best_by_ll)

In [ ]:
top_by_group = (
    successful.sort_values(["model_group", "bic"], ascending=[True, True])
    .groupby("model_group", as_index=False)
    .head(TOP_N)
    .loc[:, metric_columns]
    .reset_index(drop=True)
)
top_by_group

## Сравнение гиперпараметров

In [ ]:
hyper_summary = (
    successful.groupby(["model_group", "n_components", "covariance_type"], as_index=False)
    .agg(
        models=("random_state", "count"),
        bic_min=("bic", "min"),
        bic_median=("bic", "median"),
        bic_std=("bic", "std"),
        aic_min=("aic", "min"),
        ll_max=("log_likelihood", "max"),
        converged_rate=("converged", "mean"),
        median_seconds=("train_seconds", "median"),
    )
    .sort_values(["model_group", "bic_min"])
    .reset_index(drop=True)
)
hyper_summary

In [ ]:
groups = [group for group in ["price", "flow", "market"] if group in successful["model_group"].unique()]
fig, axes = plt.subplots(len(groups), 2, figsize=(15, 4.5 * len(groups)), constrained_layout=True, squeeze=False)

for row, group in enumerate(groups):
    group_frame = successful[successful["model_group"].eq(group)]
    for col, covariance_type in enumerate(["diag", "full"]):
        axis = axes[row][col]
        frame = group_frame[group_frame["covariance_type"].eq(covariance_type)]
        if frame.empty:
            axis.set_axis_off()
            continue
        data = [frame.loc[frame["n_components"].eq(n), "bic"].dropna().to_numpy() for n in sorted(frame["n_components"].dropna().unique())]
        labels = [str(int(n)) for n in sorted(frame["n_components"].dropna().unique())]
        axis.boxplot(data, labels=labels, showfliers=False)
        axis.set_title(f"{group} / {covariance_type}: BIC by n_components")
        axis.set_xlabel("n_components")
        axis.set_ylabel("BIC lower is better")

plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(groups), figsize=(6 * len(groups), 4.5), constrained_layout=True, squeeze=False)

for axis, group in zip(axes[0], groups):
    frame = hyper_summary[hyper_summary["model_group"].eq(group)]
    for covariance_type, color in [("diag", "#4C78A8"), ("full", "#F58518")]:
        line_frame = frame[frame["covariance_type"].eq(covariance_type)].sort_values("n_components")
        axis.plot(line_frame["n_components"], line_frame["bic_min"], marker="o", linewidth=2, color=color, label=covariance_type)
    axis.set_title(f"{group}: best BIC")
    axis.set_xlabel("n_components")
    axis.set_ylabel("min BIC")
    axis.legend()

plt.show()

## Разбор режимов лучших моделей

In [ ]:
def parse_json_array(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    if isinstance(value, str):
        return np.asarray(json.loads(value), dtype="float64")
    return np.asarray(value, dtype="float64")


def best_row(group: str, metric: str = "bic") -> pd.Series:
    frame = successful[successful["model_group"].eq(group)]
    if metric in {"bic", "aic"}:
        return frame.loc[frame[metric].idxmin()]
    return frame.loc[frame[metric].idxmax()]


selected_best = {group: best_row(group, "bic") for group in groups}
pd.DataFrame(selected_best).T.loc[:, metric_columns]

In [ ]:
fig, axes = plt.subplots(len(groups), 2, figsize=(15, 4.5 * len(groups)), constrained_layout=True, squeeze=False)

for row_idx, group in enumerate(groups):
    row = selected_best[group]
    occupancy = parse_json_array(row["state_occupancy"])
    duration = parse_json_array(row["state_mean_duration"])
    states = np.arange(len(occupancy))

    axes[row_idx][0].bar(states, occupancy, color="#4C78A8")
    axes[row_idx][0].set_title(f"{group}: state occupancy")
    axes[row_idx][0].set_xlabel("State")
    axes[row_idx][0].set_ylabel("Share")
    axes[row_idx][0].set_xticks(states)

    axes[row_idx][1].bar(states, duration, color="#F58518")
    axes[row_idx][1].set_title(f"{group}: mean regime duration")
    axes[row_idx][1].set_xlabel("State")
    axes[row_idx][1].set_ylabel("Minutes / observations")
    axes[row_idx][1].set_xticks(states)

plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(groups), figsize=(5.8 * len(groups), 5), constrained_layout=True, squeeze=False)

for axis, group in zip(axes[0], groups):
    row = selected_best[group]
    transition_matrix = parse_json_array(row["transition_matrix"])
    image = axis.imshow(transition_matrix, vmin=0, vmax=1, cmap="Blues")
    axis.set_title(f"{group}: transition matrix")
    axis.set_xlabel("To state")
    axis.set_ylabel("From state")
    ticks = np.arange(transition_matrix.shape[0])
    axis.set_xticks(ticks)
    axis.set_yticks(ticks)
    for i in range(transition_matrix.shape[0]):
        for j in range(transition_matrix.shape[1]):
            value = transition_matrix[i, j]
            axis.text(j, i, f"{value:.2f}", ha="center", va="center", color="black" if value < 0.65 else "white", fontsize=9)
    fig.colorbar(image, ax=axis, fraction=0.046, pad=0.04)

plt.show()

## Стабильность random_state для лучших структур

In [ ]:
best_structures = best_by_bic[["model_group", "n_components", "covariance_type"]].drop_duplicates()
stability_frames = []
for _, structure in best_structures.iterrows():
    mask = (
        successful["model_group"].eq(structure["model_group"])
        & successful["n_components"].eq(structure["n_components"])
        & successful["covariance_type"].eq(structure["covariance_type"])
    )
    stability_frames.append(successful.loc[mask, metric_columns])

stability = pd.concat(stability_frames, ignore_index=True).sort_values(["model_group", "bic"])
stability

In [ ]:
fig, axes = plt.subplots(1, len(groups), figsize=(6 * len(groups), 4), constrained_layout=True, squeeze=False)

for axis, group in zip(axes[0], groups):
    frame = stability[stability["model_group"].eq(group)].sort_values("random_state")
    axis.plot(frame["random_state"], frame["bic"], marker="o", linewidth=1.8)
    axis.set_title(f"{group}: BIC by random_state")
    axis.set_xlabel("random_state")
    axis.set_ylabel("BIC")

plt.show()

## Экспорт локальных таблиц

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "analysis" / "hmm_grid_search" / "outputs" / f"run_id={run_id}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

best_by_bic.to_csv(OUTPUT_DIR / "best_by_bic.csv", index=False)
best_by_aic.to_csv(OUTPUT_DIR / "best_by_aic.csv", index=False)
hyper_summary.to_csv(OUTPUT_DIR / "hyperparameter_summary.csv", index=False)
coverage.to_csv(OUTPUT_DIR / "coverage.csv", index=False)
failed.to_csv(OUTPUT_DIR / "failed_models.csv", index=False)

print(f"Saved tables to: {OUTPUT_DIR}")